# FeatFuse — Quickstart

**Feature integration for code language models.** This notebook walks through the whole
benchmark on CPU (no GPU, no model download, no JDK):

1. discover the available plugins,
2. extract engineered features from a code pair,
3. run a reproducible benchmark on the IR-Plag dataset,
4. ablate features and rank their importance,
5. read the auto-generated report, tables and figures.

It accompanies the paper *Improving Source Code Similarity Detection Through GraphCodeBERT
and Integration of Additional Features* ([arXiv:2408.08903](https://arxiv.org/abs/2408.08903)).


## 0. Install
Run once (from the repository root).


In [ ]:
# !pip install -e ".[dev]"   # uncomment if FeatFuse is not yet installed


## 1. What plugins are available?
Features, fusion strategies, encoders and datasets all live in registries.


In [ ]:
from featfuse.registry import FEATURES, FUSIONS, MODELS, DATASETS
import featfuse.features, featfuse.fusion, featfuse.models, featfuse.data  # noqa  (registers plugins)

print('features:', FEATURES.names())
print('fusion:  ', FUSIONS.names())
print('models:  ', MODELS.names())
print('datasets:', DATASETS.names())


## 2. Extract engineered features from a code pair
Each feature maps a *pair* of fragments to a scalar in (roughly) [0, 1].


In [ ]:
from featfuse.features import FeatureSet
from featfuse.types import CodePair

pair = CodePair(
    code1='public int sum(int[] a){ int s=0; for(int x: a) s+=x; return s; }',
    code2='int total(int[] xs){ int t=0; for(int i=0;i<xs.length;i++) t+=xs[i]; return t; }',
    label=1, meta={'output': 1.0})

fs = FeatureSet.from_names(['token_jaccard','char_ngram_jaccard','edit_ratio',
                            'cyclomatic_ratio','nesting_depth_ratio','exec_output_similarity'])
for name, val in zip(fs.columns, fs.vector(pair)):
    print(f'{name:24s} {val:.3f}')


## 3. Run a reproducible benchmark
One config = one fully described experiment. The classical backend runs anywhere.


In [ ]:
from featfuse.config import ExperimentConfig
from featfuse.experiment import Experiment

cfg = ExperimentConfig.from_yaml('configs/classical_features_irplag.yaml')
exp = Experiment(cfg)
results = exp.run()
for r in results['rows']:
    print(f"[{r['split']:10s}] F1={r['f1']:.3f}  MCC={r['mcc']:.3f}  ROC-AUC={r['roc_auc']:.3f}  ECE={r['ece']:.3f}")


## 4. Which features matter? (ablation + importance)


In [ ]:
ab = exp.ablate(write=False)['ablation']
loo = sorted([r for r in ab if r['setting'].startswith('-')], key=lambda r: r.get('delta_f1', 0))
print('Removing these hurts F1 the most:')
for r in loo[:3]:
    print(f"  {r['setting']:28s} delta_F1={r.get('delta_f1',0):+.4f}")

imp = exp.feature_importance()
ranked = sorted(zip(imp['columns'], imp['permutation_importance']), key=lambda t: -t[1])
print('\nTop features by permutation importance:')
for name, v in ranked[:5]:
    print(f'  {name:24s} {v:+.4f}')


## 5. The auto-generated report
Every run writes a traceable directory with tables (Markdown + LaTeX) and figures.


In [ ]:
import os
run_dir = exp.run_dir
print('artifacts:', sorted(os.listdir(run_dir)))
print()
print(open(os.path.join(run_dir, 'results_table.md')).read())


In [ ]:
from IPython.display import Image, display
fig = os.path.join(run_dir, 'figures', 'metrics.png')
display(Image(filename=fig)) if os.path.exists(fig) else print('(run the cell above first)')


## Next steps
- Add your own feature: see `docs/adding_a_feature.md`.
- Try a different fusion strategy or code model (neural backend): `docs/adding_a_fusion.md`, `docs/adding_a_model.md`.
- Reproduce the paper's neural result: `featfuse run -c configs/graphcodebert_irplag.yaml` (needs `featfuse[neural]`).

If you use FeatFuse, please cite **arXiv:2408.08903** (see `CITATION.cff`).
